In [ ]:
# 1. Total absolute wipeout of conflicting libraries
!pip uninstall -y torch torchvision torchaudio datasets fsspec gcsfs transformers evaluate accelerate

# 2. Re-install using a unified single-pass resolution layout
!pip install transformers evaluate datasets accelerate torch torchvision torchaudio

Found existing installation: torch 2.12.1
Uninstalling torch-2.12.1:
  Successfully uninstalled torch-2.12.1
Found existing installation: torchvision 0.27.1
Uninstalling torchvision-0.27.1:
  Successfully uninstalled torchvision-0.27.1
Found existing installation: torchaudio 2.11.0
Uninstalling torchaudio-2.11.0:
  Successfully uninstalled torchaudio-2.11.0
Found existing installation: datasets 5.0.0
Uninstalling datasets-5.0.0:
  Successfully uninstalled datasets-5.0.0
Found existing installation: fsspec 2026.4.0
Uninstalling fsspec-2026.4.0:
  Successfully uninstalled fsspec-2026.4.0
Found existing installation: transformers 5.12.1
Uninstalling transformers-5.12.1:
  Successfully uninstalled transformers-5.12.1
Found existing installation: evaluate 0.4.6
Uninstalling evaluate-0.4.6:
  Successfully uninstalled evaluate-0.4.6
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
  Using cached transformers-5.12.1-py3

In [ ]:
import random
import numpy as np
import torch

def set_production_seed(seed=42):
    """Secures structural reproducibility across Python, NumPy, and PyTorch matrices."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"✅ Global execution seeds anchored at: {seed}")

set_production_seed(42)

✅ Global execution seeds anchored at: 42


In [ ]:
from datasets import load_dataset

print("📥 Pulling raw dataset splits from Hugging Face...")
emotions_dataset = load_dataset("dair-ai/emotion", trust_remote_code=True)

# Define the structural mapping so the model knows what its output integers mean
id2label = {0: "sadness", 1: "joy", 2: "love", 3: "anger", 4: "fear", 5: "surprise"}
label2id = {v: k for k, v in id2label.items()}

print(f"✅ Ingested: {len(emotions_dataset['train'])} train | {len(emotions_dataset['validation'])} val lines.")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'dair-ai/emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'dair-ai/emotion' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


📥 Pulling raw dataset splits from Hugging Face...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Ingested: 16000 train | 2000 val lines.


In [ ]:
print(emotions_dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})


In [ ]:
for i in range(5):
    print(emotions_dataset['train'][i])

{'text': 'i didnt feel humiliated', 'label': 0}
{'text': 'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake', 'label': 0}
{'text': 'im grabbing a minute to post i feel greedy wrong', 'label': 3}
{'text': 'i am ever feeling nostalgic about the fireplace i will know that it is still on the property', 'label': 2}
{'text': 'i am feeling grouchy', 'label': 3}


In [ ]:
import pandas as pd

# Convert the Hugging Face train split directly to a Pandas DataFrame
df_train = emotions_dataset["train"].to_pandas()

# Inspect the first 5 rows
print("📊 PANDAS DATAFRAME PREVIEW:")
print(df_train.head())
print(f"\nDataFrame Shape: {df_train.shape}")

📊 PANDAS DATAFRAME PREVIEW:
                                                text  label
0                            i didnt feel humiliated      0
1  i can go from feeling so hopeless to so damned...      0
2   im grabbing a minute to post i feel greedy wrong      3
3  i am ever feeling nostalgic about the fireplac...      2
4                               i am feeling grouchy      3

DataFrame Shape: (16000, 2)


In [ ]:


# Add a readable string column
df_train["emotion_name"] = df_train["label"].map(id2label)

# Let's see the updated table
print(df_train[["text", "emotion_name", "label"]].head(10))

                                                text emotion_name  label
0                            i didnt feel humiliated      sadness      0
1  i can go from feeling so hopeless to so damned...      sadness      0
2   im grabbing a minute to post i feel greedy wrong        anger      3
3  i am ever feeling nostalgic about the fireplac...         love      2
4                               i am feeling grouchy        anger      3
5  ive been feeling a little burdened lately wasn...      sadness      0
6  ive been taking or milligrams or times recomme...     surprise      5
7  i feel as confused about life as a teenager or...         fear      4
8  i have been with petronas for years i feel tha...          joy      1
9                                i feel romantic too         love      2


In [ ]:
# Cuts memory usage of string columns by ~80%
df_train["emotion_name"] = df_train["emotion_name"].astype("category")
print(df_train.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   text          16000 non-null  object  
 1   label         16000 non-null  int64   
 2   emotion_name  16000 non-null  category
dtypes: category(1), int64(1), object(1)
memory usage: 266.0+ KB
None


In [ ]:
from transformers import AutoTokenizer

print("📥 Instantiating DistilBERT wordpiece tokenizer graph...")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_format_batch(batch):
    """Converts strings to integer IDs, locking context window to 128 tokens."""
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

print("⚡ Mapping tokenization across train, val, and test splits...")
tokenized_emotions = emotions_dataset.map(
    tokenize_and_format_batch,
    batched=True,
    remove_columns=["text"] # Drops raw strings from RAM to save system memory
)

print("\n🧠 PREPROCESSED TENSOR EXTREMITIES (Row 0):")
print(f"├── Input Token IDs: {tokenized_emotions['train'][0]['input_ids'][:10]}...")
print(f"└── Attention Mask:  {tokenized_emotions['train'][0]['attention_mask'][:10]}...")

📥 Instantiating DistilBERT wordpiece tokenizer graph...
⚡ Mapping tokenization across train, val, and test splits...

🧠 PREPROCESSED TENSOR EXTREMITIES (Row 0):
├── Input Token IDs: [101, 1045, 2134, 2102, 2514, 26608, 102, 0, 0, 0]...
└── Attention Mask:  [1, 1, 1, 1, 1, 1, 1, 0, 0, 0]...


In [ ]:
import numpy as np
import evaluate
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

print(f"🚀 Initializing Director Agent... Using CUDA GPU: {torch.cuda.is_available()}")

# 1. Load the model and map it explicitly to our 6 narrative emotion categories
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=6,
    id2label=id2label,
    label2id=label2id
)

# 2. Instantiate our evaluation metric tracker
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """Computes categorical prediction accuracy by identifying the maximum logit index."""
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# 3. Formulate Production-Grade Training Parameters
training_args = TrainingArguments(
    output_dir="./voxdirector_bert_agent",
    eval_strategy="epoch",            # UPDATED: Changed from evaluation_strategy to eval_strategy
    save_strategy="epoch",            # Checkpoint weight matrices at the end of each epoch
    learning_rate=2e-5,               # Step size boundary constraint
    per_device_train_batch_size=32,   # Mini-batch vector constraint size
    num_train_epochs=2,               # Complete passes over the 16,000 sentences
    weight_decay=0.01,                # L2 regularization parameter to mitigate overfitting
    logging_steps=100,                # Console reporting frequency interval
    fp16=True,                        # Activate FP16 semi-precision on Tensor Cores for speed
    report_to="none"                  # Strips unnecessary terminal telemetry noise
)

# 4. Initialize the Trainer Orchestrator
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_emotions["train"],
    eval_dataset=tokenized_emotions["validation"],
    compute_metrics=compute_metrics,
)

# 5. Launch the optimization run
print("🔥 Fine-tuning process ignited...")
trainer.train()

🚀 Initializing Director Agent... Using CUDA GPU: True


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🔥 Fine-tuning process ignited...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.266385,0.207117,0.923500
2,0.169048,0.164678,0.932500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1000, training_loss=0.3947999219894409, metrics={'train_runtime': 113.2587, 'train_samples_per_second': 282.539, 'train_steps_per_second': 8.829, 'total_flos': 1059814785024000.0, 'train_loss': 0.3947999219894409, 'epoch': 2.0})

In [ ]:
import torch
import torch.nn.functional as F

class VoxDirectorInference:
    def __init__(self, trained_model, dataset_tokenizer, labels_map):
        """Binds the trained weights, wordpiece tokenizer, and class labels."""
        self.model = trained_model.to("cuda" if torch.cuda.is_available() else "cpu")
        self.tokenizer = dataset_tokenizer
        self.labels_map = labels_map
        self.model.eval() # Sets layers like dropout to evaluation mode

    def predict_direction(self, raw_text: str):
        """Tokenizes incoming novel sentences and returns the primary emotion and confidence score."""
        # 1. Standard deterministic tokenization matching our training shape
        inputs = self.tokenizer(
            raw_text,
            return_tensors="pt",
            truncation=True,
            padding="max_length",
            max_length=128
        )

        # Move inputs to the active GPU space
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}

        # 2. Run forward pass through the model without calculating gradients
        with torch.no_grad():
            outputs = self.model(**inputs)

        # 3. Extract unnormalized logits and convert them to stable probabilities via Softmax
        probabilities = F.softmax(outputs.logits, dim=-1).squeeze().tolist()

        # 4. Map back to human-readable labels
        prediction_idx = np.argmax(probabilities)
        assigned_emotion = self.labels_map[prediction_idx]
        confidence_score = probabilities[prediction_idx]

        return assigned_emotion, confidence_score, probabilities

# Initialize our inference agent using the active weights and tokenizer
director_agent = VoxDirectorInference(model, tokenizer, id2label)
print("🎯 Inference engine initialized and standing by for narration lines!")

🎯 Inference engine initialized and standing by for narration lines!


In [ ]:
import numpy as np
import evaluate
from sklearn.metrics import classification_report, confusion_matrix

print("📥 Running Comprehensive Test Set Evaluation...")

# 1. Generate predictions across the unseen 2,000 test records
test_predictions = trainer.predict(tokenized_emotions["test"])
logits = test_predictions.predictions
true_labels = test_predictions.label_ids

# 2. Convert raw logits to predicted class indices
predicted_indices = np.argmax(logits, axis=-1)

# 3. Map numerical IDs back to readable emotion strings
target_names = [id2label[i] for i in range(6)]

# 4. Generate the complete MLE Classification Report
report = classification_report(
    true_labels,
    predicted_indices,
    target_names=target_names,
    digits=4 # Show precision to 4 decimal places
)

print("\n🏆 ENTERPRISE CLASSIFICATION REPORT:")
print(report)

# 5. Output the raw Confusion Matrix array
conf_matrix = confusion_matrix(true_labels, predicted_indices)
print("\n🔍 CONFUSION MATRIX (Rows: True, Columns: Predicted):")
print(conf_matrix)

📥 Running Comprehensive Test Set Evaluation...



🏆 ENTERPRISE CLASSIFICATION REPORT:
              precision    recall  f1-score   support

     sadness     0.9601    0.9535    0.9568       581
         joy     0.9413    0.9468    0.9440       695
        love     0.8171    0.8428    0.8297       159
       anger     0.9164    0.9164    0.9164       275
        fear     0.8571    0.9107    0.8831       224
    surprise     0.8085    0.5758    0.6726        66

    accuracy                         0.9200      2000
   macro avg     0.8834    0.8576    0.8671      2000
weighted avg     0.9197    0.9200    0.9191      2000


🔍 CONFUSION MATRIX (Rows: True, Columns: Predicted):
[[554   6   1  11   9   0]
 [  2 658  29   3   0   3]
 [  0  24 134   1   0   0]
 [ 12   3   0 252   8   0]
 [  6   0   0   8 204   6]
 [  3   8   0   0  17  38]]


In [ ]:
# CELL 7: ADVANCED NARRATIVE STRESS-TEST

stress_test_lines = [
    # 1. Atmospheric Fear (No explicit "I am scared" words)
    "The floorboards creaked overhead. She held her breath, pressing her back against the closet door.",

    # 2. Quiet, Melancholic Sadness
    "He carefully packed her favorite books into the cardboard box, knowing the house would be empty tomorrow.",

    # 3. Sudden Surprise
    "The detective pulled back the heavy velvet curtain and gasped. The vault was completely empty!",

    # 4. Pure Romantic Love
    "She looked into his eyes and smiled, realizing that after all these years, he was still her safe place.",

    # 5. High-Octane Anger
    "How dare you lie to my face after everything this family has sacrificed for you!",

    # 6. Ambiguous Neutral / Descriptive (Should yield low confidence across all classes)
    "The grandfather clock stood in the hallway, ticking quietly as the afternoon light shifted across the rug."
]

print("🔬 STRESS-TESTING NARRATIVE EDGE CASES:")
print("=" * 65)

for line in stress_test_lines:
    emotion, confidence, all_probs = director_agent.predict_direction(line)

    # Flag low confidence predictions for backend fallback routing
    status_flag = "⚠️ FALLBACK TO NEUTRAL" if confidence < 0.70 else "✅ ACCEPTED"

    print(f"\n📖 Text: '{line}'")
    print(f"├── 🎙️ Primary Direction: [{emotion.upper()}] ({confidence*100:.1f}%)")
    print(f"└── 🔀 System Routing:    {status_flag}")

🔬 STRESS-TESTING NARRATIVE EDGE CASES:

📖 Text: 'The floorboards creaked overhead. She held her breath, pressing her back against the closet door.'
├── 🎙️ Primary Direction: [FEAR] (85.6%)
└── 🔀 System Routing:    ✅ ACCEPTED

📖 Text: 'He carefully packed her favorite books into the cardboard box, knowing the house would be empty tomorrow.'
├── 🎙️ Primary Direction: [SADNESS] (93.4%)
└── 🔀 System Routing:    ✅ ACCEPTED

📖 Text: 'The detective pulled back the heavy velvet curtain and gasped. The vault was completely empty!'
├── 🎙️ Primary Direction: [SADNESS] (45.6%)
└── 🔀 System Routing:    ⚠️ FALLBACK TO NEUTRAL

📖 Text: 'She looked into his eyes and smiled, realizing that after all these years, he was still her safe place.'
├── 🎙️ Primary Direction: [JOY] (98.6%)
└── 🔀 System Routing:    ✅ ACCEPTED

📖 Text: 'How dare you lie to my face after everything this family has sacrificed for you!'
├── 🎙️ Primary Direction: [ANGER] (55.4%)
└── 🔀 System Routing:    ⚠️ FALLBACK TO NEUTRAL

📖 Text

In [ ]:
# CELL 9: THE PRISTINE DEPLOYMENT EXPORT

import os
import shutil
from google.colab import files

print("🧹 Assembling exact production bundle...")

# 1. Create a dedicated clean folder just for our 5 deployment files
os.makedirs("production_bundle", exist_ok=True)

# 2. Grab the Brain (from checkpoint-1000)
shutil.copy("./voxdirector_bert_agent/checkpoint-1000/model.safetensors", "production_bundle/model.safetensors")
shutil.copy("./voxdirector_bert_agent/checkpoint-1000/config.json", "production_bundle/config.json")

# 3. Force the Tokenizer to explicitly save its 3 text files directly into our folder
tokenizer.save_pretrained("production_bundle")

# 4. Verify the exact file count inside Colab before downloading
bundled_files = os.listdir("production_bundle")
print(f"\n📦 VERIFIED FILES IN BUNDLE ({len(bundled_files)}):")
for f in bundled_files:
    print(f" ├── {f}")

# 5. Zip and Download
shutil.make_archive("voxdirector_production_bundle", "zip", "production_bundle")
print("\n⬇️ Downloading guaranteed bundle to your browser...")
files.download("voxdirector_production_bundle.zip")

🧹 Assembling exact production bundle...

📦 VERIFIED FILES IN BUNDLE (4):
 ├── config.json
 ├── tokenizer_config.json
 ├── tokenizer.json
 ├── model.safetensors

⬇️ Downloading guaranteed bundle to your browser...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>